In [1]:
# Importing Modules
import pandas as pd
from sklearn.model_selection import train_test_split
from src.utils.PreprocessSMILES import preprocess
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [2]:
# Download all datasets
from src.utils.download import download_all
download_all()

In [3]:
# Loading ESOL data
esol_data = pd.read_csv("data/ESOL.csv")[["smiles", "measured log solubility in mols per litre"]]
esol_data = esol_data.rename(columns={"measured log solubility in mols per litre":"target"})

# Loading Lipophilicity data
lipophil_data = pd.read_csv("data/Lipophilicity.csv")[["smiles", "exp"]]
lipophil_data= lipophil_data.rename(columns={"exp":"target"})

# Loading retention time (RT) data
rt_data = pd.read_csv("data/RT.csv")
rt_data= rt_data.rename(columns={"SMILES":"smiles","RTs":"target"})

# Loading B3DB data
b3db_data = pd.read_csv("data/B3DB.tsv", sep="\t")[["SMILES", "logBB"]]
b3db_data = b3db_data.rename(columns={"SMILES":"smiles", "logBB":"target"})

In [4]:
# Preprocess ESOL data
esol_data = preprocess(esol_data)

# Preprocess Lipophilicity data
lipophil_data = preprocess(lipophil_data)

# Preprocess retention time (RT) data
rt_data = preprocess(rt_data)

# Preprocess B3DB data
b3db_data = preprocess(b3db_data)

In [5]:
# Subset ESOL data
esol_data = esol_data.sample(1000, random_state=42).reset_index(drop=True)

# Subset Lipophilicity data
lipophil_data = lipophil_data.sample(1000, random_state=42).reset_index(drop=True)

# Subset retention time (RT) data
rt_data = rt_data.sample(1000, random_state=42).reset_index(drop=True)

# Subset B3DB data
b3db_data = b3db_data.sample(1000, random_state=42).reset_index(drop=True)

In [6]:
# Dict
avail_data = {"ESOL":esol_data, "Lipophilicity":lipophil_data, "RT":rt_data, "B3DB":b3db_data}

In [7]:
for dataset in avail_data.keys():
    # Raw dataset
    X = avail_data[dataset]["smiles"].to_numpy()
    y = avail_data[dataset]["target"].to_numpy()
    # Dividing dataset
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=100, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=100, random_state=42)
    # Merging X and y
    train = pd.DataFrame({"smiles":X_train, "target":y_train})
    val = pd.DataFrame({"smiles":X_val, "target":y_val})
    test = pd.DataFrame({"smiles":X_test, "target":y_test})
    # Saving files
    avail_data[dataset].to_csv(f"data/processed/{dataset}.csv", quoting=False, index=False)
    train.to_csv(f"data/processed/train/{dataset}.csv", quoting=False, index=False)
    val.to_csv(f"data/processed/val/{dataset}.csv", quoting=False, index=False)
    test.to_csv(f"data/processed/test/{dataset}.csv", quoting=False, index=False)